# DART 기업 요약 재무분석 v1

티커 입력 → 네이버 금융 '기업실적분석' 형태의 요약 + 영업활동 흐름 진단 + DuPont 분해를
생성하고 Excel/CSV 로 저장하는 노트북 (AI 분석 입력용).

**출력 (시트 5개):**
1. ①연간요약 — 매출·영업이익·순이익·이익률·ROE·부채비율 (4분기 완비 연도만)
2. ②분기요약 — 위 항목의 분기 시계열 + 당좌비율·유보율 (억원/%)
3. ③회전일수TTM — 매출TTM, COGS TTM, DSO, DIO, DPO, CCC (요청 양식)
4. ④DuPont — 3단(순이익률×회전율×레버리지) + 5단(세금·이자부담·영업이익률) TTM 분해
5. ⑤원데이터 — 표준 wide 원본 (원 단위, 검산용)

**계산 규약:** flow 는 4분기 연속 TTM(결측 분기 있으면 NaN), 잔액 비율은 기말,
평균잔액(ROE·회전율)은 (기말+4분기전)/2. 자본총계 미매핑 기업은 자산총계−부채총계로 자동 대체.

In [21]:
# ═══════════════════════════════════════════════════════════════
#  ★ 입력 변수 — 이 셀만 수정하세요
# ═══════════════════════════════════════════════════════════════
TICKER      = "000660"        # 분석 대상 (6자리, 'A' 접두어 허용)
N_QUARTERS  = 20              # 최근 5년 = 20분기
SAVE_FORMAT = "xlsx"          # None | "xlsx" | "csv"
OUTPUT_DIR  = r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\Fundamental_Analysis"   # 저장 폴더 (예: r"D:\dart_out")

DB_INFO = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
TABLE_DART_FS = "korea_fs_data_from_DART_V2"
VERBOSE = True
print(f"[OK] 대상={TICKER}  기간={N_QUARTERS}분기  저장={SAVE_FORMAT} → {OUTPUT_DIR}")


[OK] 대상=000660  기간=20분기  저장=xlsx → C:\Users\82108\OneDrive\INVESTMENT\한국주식\Fundamental_Analysis


## Cell 2 · DART → 표준 wide 변환 (v1.2 — 요약분석 필드 확장판)

In [22]:
# ═══════════════════════════════════════════════════════════════
#  DART long → 표준 wide 변환 모듈
#  - account_id 우선 매칭, 실패 시 account_nm 정규식 fallback
#  - IS/CIS: 누적 vs 3개월 자동 감지 후 분기화 (Q4 = FY − 3개분기)
#  - CF: 항상 누적으로 간주 → 차분 (Q2=H1−Q1, Q3=Q3−H1, Q4=FY−Q3)
#  - BS: 시점 잔액 그대로
# ═══════════════════════════════════════════════════════════════
import re
import numpy as np
import pandas as pd
import pymysql
from datetime import datetime


def log(tag, msg):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)


def norm_ticker(t: str) -> str:
    """'A005930' → '005930'"""
    t = str(t).strip().upper()
    return t[1:].zfill(6) if t.startswith("A") else t.zfill(6)


def to_dg_ticker(t: str) -> str:
    """'005930' → 'A005930' (forecast 테이블용)"""
    return "A" + norm_ticker(t)


# ───────────────────────────────────────────────────────────────
#  표준 필드 매핑 정의
#    ids : account_id 후보 (정확 일치, 접두어 ifrs_/ifrs-full_ 모두 등록)
#    nm  : account_nm 정규식 후보 (앞에 있을수록 우선)
#    sj  : 허용 재무제표 (앞에 있을수록 우선; IS 우선, 없으면 CIS)
#    agg : 'pick'=대표 계정 1개 선택(중복합산 방지) / 'sum'=매칭 계정 전부 합산
# ───────────────────────────────────────────────────────────────
def _ids(*stems):
    out = []
    for s in stems:
        out += [f"ifrs_{s}", f"ifrs-full_{s}"]
    return out


FIELD_MAP = {
    # ── 손익 (flow) ──
    "revenue": dict(
        ids=_ids("Revenue") + ["dart_Revenue"],
        nm=[r"^매출액$", r"^매출$", r"^수익\(매출액\)$", r"^영업수익$"],
        sj=["IS", "CIS"], agg="pick"),
    "operating_income": dict(
        ids=["dart_OperatingIncomeLoss"] + _ids("OperatingIncomeLoss"),
        nm=[r"^영업이익", r"^영업손익"],
        sj=["IS", "CIS"], agg="pick"),
    "pretax_income": dict(
        ids=_ids("ProfitLossBeforeTax"),
        nm=[r"법인세비용차감전", r"^세전.*이익"],
        sj=["IS", "CIS"], agg="pick"),
    "tax_expense": dict(
        ids=_ids("IncomeTaxExpenseContinuingOperations", "IncomeTaxExpense"),
        nm=[r"^법인세비용"],
        sj=["IS", "CIS"], agg="pick"),
    "interest_expense": dict(
        ids=_ids("FinanceCosts") + ["dart_InterestExpenseFinanceExpense"],
        nm=[r"^이자비용", r"^금융비용"],
        sj=["IS", "CIS"], agg="pick"),

    # ── 현금흐름 (flow, 누적) ──
    "da_cf": dict(
        ids=["dart_AdjustmentsForDepreciationExpense"] +
            _ids("AdjustmentsForDepreciationExpense",
                 "AdjustmentsForDepreciationAndAmortisationExpense",
                 "DepreciationAndAmortisationExpense"),
        nm=[r"감가상각비와\s*무형자산상각", r"감가상각비\s*및\s*상각",
            r"감가상각"],                    # 앞머리 고정 제거 — "유형자산 감가상각비" 등 대응
        sj=["CF"], agg="pick"),
    "intangible_amort_cf": dict(
        ids=["dart_AmortisationExpense"] +
            _ids("AdjustmentsForAmortisationExpense", "AmortisationExpense"),
        nm=[r"^(?!.*감가상각).*무형자산\s*상각"],   # 합산계정(감가상각비와 무형자산상각비) 제외 — da_cf 와 이중계상 방지
        sj=["CF"], agg="pick"),
    "capex_tangible": dict(
        ids=_ids("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
                 "PurchaseOfPropertyPlantAndEquipment"),
        nm=[r"유형자산의\s*취득", r"유형자산의\s*증가", r"유형자산\s*취득",
            r"토지.*취득|건설중인자산.*(?:취득|증가)"],
        sj=["CF"], agg="pick"),
    "capex_intangible": dict(
        ids=_ids("PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
                 "PurchaseOfIntangibleAssets"),
        nm=[r"무형자산의\s*취득", r"무형자산의\s*증가", r"무형자산\s*취득"],
        sj=["CF"], agg="pick"),

    # ── 재무상태 (stock) ──
    "receivables": dict(
        ids=_ids("TradeAndOtherCurrentReceivables", "CurrentTradeReceivables"),
        nm=[r"^매출채권$", r"^매출채권\s*및", r"^매출채권과"],
        sj=["BS"], agg="pick"),
    "inventories": dict(
        ids=_ids("Inventories"),
        nm=[r"^재고자산"],
        sj=["BS"], agg="pick"),
    "prepaid_expenses": dict(
        ids=[], nm=[r"^선급비용"], sj=["BS"], agg="pick"),
    "payables": dict(
        ids=_ids("TradeAndOtherCurrentPayables", "CurrentTradePayables"),
        nm=[r"^매입채무$", r"^매입채무\s*및", r"^매입채무와"],
        sj=["BS"], agg="pick"),
    "accrued_expenses": dict(
        ids=[], nm=[r"^미지급비용"], sj=["BS"], agg="pick"),
    "other_payables": dict(
        ids=[], nm=[r"^미지급금"], sj=["BS"], agg="pick"),
    "advances_received": dict(
        ids=[], nm=[r"^선수금"], sj=["BS"], agg="pick"),
    "contract_liabilities": dict(
        ids=_ids("ContractLiabilities"),
        nm=[r"^계약부채"], sj=["BS"], agg="pick"),

    "short_term_debt": dict(
        ids=["dart_ShortTermBorrowings"] + _ids("ShorttermBorrowings"),
        nm=[r"^단기차입금"], sj=["BS"], agg="pick"),
    "current_lt_debt": dict(
        ids=[], nm=[r"^유동성장기부채", r"^유동성장기차입금", r"^유동성사채"],
        sj=["BS"], agg="sum"),
    "bonds": dict(
        ids=[], nm=[r"^사채$", r"^사채\("], sj=["BS"], agg="pick"),
    "long_term_debt": dict(
        ids=["dart_LongTermBorrowingsGross"],
        nm=[r"^장기차입금"], sj=["BS"], agg="pick"),
    "lease_liab": dict(
        ids=_ids("LeaseLiabilities", "CurrentLeaseLiabilities",
                 "NoncurrentLeaseLiabilities"),
        nm=[r"리스부채"], sj=["BS"], agg="sum"),

    "cash": dict(
        ids=_ids("CashAndCashEquivalents"),
        nm=[r"^현금및현금성자산"], sj=["BS"], agg="pick"),
    "short_term_invest": dict(
        ids=["dart_ShortTermDepositsNotClassifiedAsCashEquivalents"],
        nm=[r"^단기금융상품", r"^단기투자자산"], sj=["BS"], agg="pick"),
    "total_equity": dict(
        ids=_ids("Equity"),
        nm=[r"^자본총계"], sj=["BS"], agg="pick"),
    "cogs": dict(
        ids=_ids("CostOfSales"),
        nm=[r"^매출원가"], sj=["IS", "CIS"], agg="pick"),
    "net_income": dict(
        ids=_ids("ProfitLoss"),
        nm=[r"^당기순이익", r"^분기순이익", r"^반기순이익", r"^연결당기순이익",
            r"당기순이익\(손실\)", r"^순이익"],
        sj=["IS", "CIS"], agg="pick"),
    "total_liabilities": dict(
        ids=_ids("Liabilities"),
        nm=[r"^부채총계"], sj=["BS"], agg="pick"),
    "current_assets": dict(
        ids=_ids("CurrentAssets"),
        nm=[r"^유동자산"], sj=["BS"], agg="pick"),
    "current_liabilities": dict(
        ids=_ids("CurrentLiabilities"),
        nm=[r"^유동부채"], sj=["BS"], agg="pick"),
    "issued_capital": dict(
        ids=_ids("IssuedCapital"),
        nm=[r"^자본금"], sj=["BS"], agg="pick"),
    "retained_earnings": dict(
        ids=_ids("RetainedEarnings"),
        nm=[r"^이익잉여금"], sj=["BS"], agg="pick"),
    "capital_surplus": dict(
        ids=["dart_CapitalSurplus"] + _ids("SharePremium"),
        nm=[r"^자본잉여금", r"^주식발행초과금"], sj=["BS"], agg="pick"),
    "ppe": dict(
        ids=_ids("PropertyPlantAndEquipment"),
        nm=[r"^유형자산$"], sj=["BS"], agg="pick"),
    "intangible_assets": dict(
        ids=_ids("IntangibleAssetsOtherThanGoodwill", "IntangibleAssets"),
        nm=[r"^무형자산$"], sj=["BS"], agg="pick"),
    "total_assets": dict(
        ids=_ids("Assets"),
        nm=[r"^자산총계"], sj=["BS"], agg="pick"),
}

QUARTER_ORDER = {"Q1": 1, "H1": 2, "Q3": 3, "FY": 4}
FLOW_SJ  = {"IS", "CIS", "CF"}


def _load_ticker_long(ticker: str, db_info: dict, table: str) -> pd.DataFrame:
    conn = pymysql.connect(**db_info, charset="utf8mb4")
    try:
        df = pd.read_sql(
            f"""SELECT bsns_year, quarter, sj_div, account_id, account_nm,
                       thstrm_amount, report_date
                FROM {table} WHERE ticker = %s""",
            conn, params=[norm_ticker(ticker)])
    finally:
        conn.close()
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"])
    return df


def _match_field(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    """
    필드 정의(spec)에 맞는 행 선택. 반환: (bsns_year, quarter) 별 단일 값.

    ★ v1.1: 기간별 선택(per-period pick)으로 변경.
      K-IFRS 분류체계가 2019년경 ifrs_ → ifrs-full_ 로 전환되어 같은 항목이
      기간별로 다른 account_id 로 쪼개져 있음. 대표 계정 1개만 고르면
      전환 이전/이후 한쪽 히스토리가 통째로 사라지므로, (연도,분기)마다
      우선순위(id 순서 > nm 패턴 순서 > 커버리지)가 가장 높은 행 1개를 선택.
      같은 분기에 유사 계정이 중복돼도 1개만 뽑아 이중계상은 여전히 차단됨.
    """
    sub = df[df["sj_div"].isin(spec["sj"])].copy()
    if sub.empty:
        return pd.DataFrame()

    # 후보 수집 + 우선순위(rank) 부여: id 일치(0~) < nm 패턴(1000~)
    id_rank = {aid: i for i, aid in enumerate(spec["ids"])}
    cand = sub[sub["account_id"].isin(id_rank)].copy()
    if not cand.empty:
        cand["rank"] = cand["account_id"].map(id_rank)
    if spec["nm"]:
        rest = sub.drop(index=cand.index) if not cand.empty else sub
        parts = [cand] if not cand.empty else []
        for j, pat in enumerate(spec["nm"]):
            m = rest[rest["account_nm"].astype(str).str.strip()
                        .str.contains(pat, regex=True, na=False)]
            if not m.empty:
                m = m.copy()
                m["rank"] = 1000 + j
                parts.append(m)
                rest = rest.drop(index=m.index)
        cand = pd.concat(parts) if parts else cand
    if cand.empty:
        return pd.DataFrame()

    # sj_div 우선순위 (IS > CIS 등): 상위 sj에 데이터가 있으면 그것만
    for sj in spec["sj"]:
        c2 = cand[cand["sj_div"] == sj]
        if not c2.empty:
            cand = c2
            break

    if spec["agg"] == "sum":
        # 같은 분기의 서로 다른 계정을 합산 (리스부채 유동+비유동 등)
        out = (cand.groupby(["bsns_year", "quarter"], as_index=False)
                   ["thstrm_amount"].sum(min_count=1))
    else:
        # per-period pick: 분기별로 rank 최소(동률이면 전체 커버리지 최대) 행 1개
        cov = cand.groupby("account_id")["bsns_year"].count()
        cand["cov"] = cand["account_id"].map(cov)
        cand = cand.sort_values(["rank", "cov"], ascending=[True, False])
        out = cand.drop_duplicates(subset=["bsns_year", "quarter"], keep="first")[
            ["bsns_year", "quarter", "thstrm_amount"]].copy()
    return out


def _detect_cumulative(pivot: pd.DataFrame) -> bool:
    """
    IS 계열 flow 가 누적인지 3개월치인지 자동 감지.
    (Q1+H1+Q3)/FY 중앙값: 3개월치 ≈ 0.75, 누적 ≈ 1.5 → 임계 1.1
    """
    ratios = []
    for y, row in pivot.iterrows():
        if all(pd.notnull(row.get(q)) for q in ("Q1", "H1", "Q3", "FY")) \
                and row["FY"] not in (0, None):
            ratios.append((row["Q1"] + row["H1"] + row["Q3"]) / row["FY"])
    if not ratios:
        return False   # 판단 불가 → 3개월치 가정 (보수적)
    return float(np.median(ratios)) > 1.1


def _flow_to_quarterly(series_df: pd.DataFrame, force_cumulative: bool = None):
    """
    flow 항목 (연도,분기,값) → 분기화 값 dict {(year,'Qn'): value}.
    force_cumulative: None=자동감지, True=누적 차분, False=3개월치 취급
    반환: (dict, cumulative여부)
    """
    pivot = series_df.pivot_table(index="bsns_year", columns="quarter",
                                  values="thstrm_amount", aggfunc="first")
    cum = _detect_cumulative(pivot) if force_cumulative is None else force_cumulative

    out = {}
    for y, row in pivot.iterrows():
        q1, h1, q3, fy = (row.get("Q1"), row.get("H1"),
                          row.get("Q3"), row.get("FY"))
        if cum:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1 - q1 if pd.notnull(h1) and pd.notnull(q1) else np.nan
            out[(y, "Q3")] = q3 - h1 if pd.notnull(q3) and pd.notnull(h1) else np.nan
            out[(y, "Q4")] = fy - q3 if pd.notnull(fy) and pd.notnull(q3) else np.nan
        else:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1
            out[(y, "Q3")] = q3
            if all(pd.notnull(v) for v in (fy, q1, h1, q3)):
                out[(y, "Q4")] = fy - (q1 + h1 + q3)
            else:
                out[(y, "Q4")] = np.nan
    return out, cum


_QDATE = {"Q1": "-03-31", "Q2": "-06-30", "Q3": "-09-30", "Q4": "-12-31"}


def load_dart_financials_wide(ticker: str, db_info: dict,
                              table_name: str = None,
                              item_keys=None, fillna_zero: bool = False,
                              verbose: bool = False) -> pd.DataFrame:
    """
    DART long 테이블 → 분기 wide DataFrame (index=분기말 date, 단위=원).
    기존 load_korea_financials_wide 와 동일한 사용 패턴.
    """
    table_name = table_name or TABLE_DART_FS
    raw = _load_ticker_long(ticker, db_info, table_name)
    if raw.empty:
        return pd.DataFrame()

    fields = item_keys or list(FIELD_MAP.keys())
    col_data, cum_info = {}, {}

    for f in fields:
        spec = FIELD_MAP[f]
        sel = _match_field(raw, spec)
        if sel.empty:
            continue
        if spec["sj"][0] in FLOW_SJ:
            force = True if spec["sj"] == ["CF"] else None   # CF는 항상 누적
            qvals, cum = _flow_to_quarterly(sel, force_cumulative=force)
            cum_info[f] = cum
        else:  # BS: 시점 잔액, H1→Q2 라벨만 변경
            qvals = {}
            for _, r in sel.iterrows():
                q = {"Q1": "Q1", "H1": "Q2", "Q3": "Q3", "FY": "Q4"}[r["quarter"]]
                qvals[(int(r["bsns_year"]), q)] = r["thstrm_amount"]
        col_data[f] = qvals

    if not col_data:
        return pd.DataFrame()

    all_keys = sorted({k for v in col_data.values() for k in v})
    idx = pd.to_datetime([f"{y}{_QDATE[q]}" for y, q in all_keys])
    wide = pd.DataFrame(
        {f: [col_data[f].get(k, np.nan) for k in all_keys] for f in col_data},
        index=idx).sort_index()

    if fillna_zero:
        wide = wide.fillna(0.0)

    if verbose:
        cum_flows = [f for f, c in cum_info.items() if c]
        log(norm_ticker(ticker),
            f"wide shape={wide.shape} 기간={wide.index.min().date()}~{wide.index.max().date()}"
            + (f"  누적차분 적용: {cum_flows}" if cum_flows else ""))
    return wide


print("[OK] DART wide 변환 모듈 로드 완료")


[OK] DART wide 변환 모듈 로드 완료


## Cell 3 · 요약분석 빌더

In [23]:
# ═══════════════════════════════════════════════════════════════
#  기업 요약 재무분석 — company_summary(ticker)
#   ① 연간 요약 (네이버 기업실적분석 형태)
#   ② 분기 요약 (최근 N분기)
#   ③ 회전일수 TTM (매출TTM, COGS TTM, DSO, DIO, DPO, CCC)
#   ④ DuPont TTM (3단 + 5단 분해)
#   → SAVE_FORMAT 에 따라 xlsx(멀티시트) 또는 csv 로 OUTPUT_DIR 에 저장
# ═══════════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
from datetime import datetime


def _qlabel(idx: pd.DatetimeIndex) -> list:
    return [f"{d.year}Q{(d.month - 1)//3 + 1}" for d in idx]


def _prep_wide(ticker: str) -> pd.DataFrame:
    """wide 로드 + 분기 연속 리인덱스 + 자본총계 fallback."""
    w = load_dart_financials_wide(ticker, DB_INFO, TABLE_DART_FS)
    if w.empty:
        raise ValueError(f"[{ticker}] DART 데이터 없음")
    # 분기말 연속 grid 로 리인덱스 (rolling TTM 이 구멍을 건너뛰지 않도록)
    full = pd.date_range(w.index.min(), w.index.max(), freq="QE")
    w = w.reindex(full)

    # ★ 자본총계 fallback: 자산총계 − 부채총계 (자본총계 계정 매핑 실패 기업 대응)
    if "total_equity" not in w.columns:
        w["total_equity"] = np.nan
    if {"total_assets", "total_liabilities"}.issubset(w.columns):
        est = w["total_assets"] - w["total_liabilities"]
        w["total_equity"] = w["total_equity"].fillna(est)
    return w


def _ttm(s: pd.Series) -> pd.Series:
    """4분기 연속 합 (하나라도 결측이면 NaN — 왜곡 방지)."""
    return s.rolling(4, min_periods=4).sum()


def _avg_bal(s: pd.Series, lag: int = 4) -> pd.Series:
    """기초(1년 전)·기말 평균 잔액."""
    return (s + s.shift(lag)) / 2


def build_turnover_ttm(w: pd.DataFrame) -> pd.DataFrame:
    """③ 회전일수 진단 — TTM 기준 (요청 양식)."""
    rev_ttm = _ttm(w.get("revenue", pd.Series(np.nan, index=w.index)))
    cogs_ttm = _ttm(w.get("cogs", pd.Series(np.nan, index=w.index)))

    recv = w.get("receivables")
    inv = w.get("inventories")
    pay = w.get("payables")

    out = pd.DataFrame(index=w.index)
    out["매출 TTM"] = rev_ttm
    out["COGS TTM"] = cogs_ttm
    out["DSO(일)"] = (recv / rev_ttm * 365) if recv is not None else np.nan
    out["DIO(일)"] = (inv / cogs_ttm * 365) if inv is not None else np.nan
    out["DPO(일)"] = (pay / cogs_ttm * 365) if pay is not None else np.nan
    out["CCC(일)"] = out["DSO(일)"] + out["DIO(일)"] - out["DPO(일)"]
    out.insert(0, "Quarter", _qlabel(out.index))
    return out


def build_dupont_ttm(w: pd.DataFrame) -> pd.DataFrame:
    """④ DuPont — ROE 원천 분해 (TTM).
    3단: ROE = 순이익률 × 총자산회전율 × 재무레버리지
    5단: ROE = 세금부담 × (세전/영업) × 영업이익률 × 회전율 × 레버리지
    """
    ni = _ttm(w.get("net_income", pd.Series(np.nan, index=w.index)))
    rev = _ttm(w.get("revenue", pd.Series(np.nan, index=w.index)))
    ebt = _ttm(w.get("pretax_income", pd.Series(np.nan, index=w.index)))
    oi = _ttm(w.get("operating_income", pd.Series(np.nan, index=w.index)))
    avg_a = _avg_bal(w.get("total_assets", pd.Series(np.nan, index=w.index)))
    avg_e = _avg_bal(w.get("total_equity", pd.Series(np.nan, index=w.index)))

    out = pd.DataFrame(index=w.index)
    out["순이익률(%)"] = ni / rev * 100
    out["총자산회전율(회)"] = rev / avg_a
    out["재무레버리지(배)"] = avg_a / avg_e
    out["ROE(%)"] = ni / avg_e * 100
    # 5단 분해
    out["세금부담(NI/EBT)"] = ni / ebt
    out["이자·기타부담(EBT/OI)"] = ebt / oi
    out["영업이익률(%)"] = oi / rev * 100
    out["ROA(%)"] = ni / avg_a * 100
    out.insert(0, "Quarter", _qlabel(out.index))
    return out.replace([np.inf, -np.inf], np.nan)


def build_quarterly_summary(w: pd.DataFrame) -> pd.DataFrame:
    """② 분기 요약 (네이버 '최근 분기 실적' 형태 + 확장, 단위: 억원/%)"""
    e8 = 1e8
    ni_ttm = _ttm(w.get("net_income", pd.Series(np.nan, index=w.index)))
    avg_e = _avg_bal(w.get("total_equity", pd.Series(np.nan, index=w.index)))

    out = pd.DataFrame(index=w.index)
    out["매출액(억원)"] = w.get("revenue") / e8
    out["영업이익(억원)"] = w.get("operating_income") / e8
    out["당기순이익(억원)"] = w.get("net_income") / e8
    out["영업이익률(%)"] = w.get("operating_income") / w.get("revenue") * 100
    out["순이익률(%)"] = w.get("net_income") / w.get("revenue") * 100
    out["ROE(%)_TTM"] = ni_ttm / avg_e * 100
    if {"total_liabilities", "total_equity"}.issubset(w.columns):
        out["부채비율(%)"] = w["total_liabilities"] / w["total_equity"] * 100
    if {"current_assets", "inventories", "current_liabilities"}.issubset(w.columns):
        out["당좌비율(%)"] = ((w["current_assets"] - w["inventories"].fillna(0))
                          / w["current_liabilities"] * 100)
    if {"retained_earnings", "issued_capital"}.issubset(w.columns):
        surplus = w["retained_earnings"].fillna(0) + \
                  w.get("capital_surplus", pd.Series(0, index=w.index)).fillna(0)
        out["유보율(%)"] = surplus / w["issued_capital"] * 100
    out.insert(0, "Quarter", _qlabel(out.index))
    return out.replace([np.inf, -np.inf], np.nan)


def build_annual_summary(w: pd.DataFrame) -> pd.DataFrame:
    """① 연간 요약 — 4분기 완비 연도만 (단위: 억원/%)"""
    e8 = 1e8
    g = w.groupby(w.index.year)
    rev = g["revenue"].agg(["sum", "count"]) if "revenue" in w.columns else None
    rows = []
    for y, grp in g:
        n_rev = grp["revenue"].notna().sum() if "revenue" in grp.columns else 0
        if n_rev < 4:
            continue
        r = {"연도": y,
             "매출액(억원)": grp["revenue"].sum() / e8,
             "영업이익(억원)": grp.get("operating_income", pd.Series(dtype=float)).sum() / e8,
             "당기순이익(억원)": grp.get("net_income", pd.Series(dtype=float)).sum() / e8}
        r["영업이익률(%)"] = r["영업이익(억원)"] / r["매출액(억원)"] * 100 if r["매출액(억원)"] else np.nan
        r["순이익률(%)"] = r["당기순이익(억원)"] / r["매출액(억원)"] * 100 if r["매출액(억원)"] else np.nan
        # 연말 잔액 기반 비율
        end = grp.iloc[-1]
        eq_end = end.get("total_equity", np.nan)
        eq_beg = w["total_equity"].shift(4).reindex(grp.index).iloc[-1] \
            if "total_equity" in w.columns else np.nan
        avg_eq = np.nanmean([eq_end, eq_beg])
        r["ROE(%)"] = (r["당기순이익(억원)"] * e8) / avg_eq * 100 if avg_eq and avg_eq > 0 else np.nan
        if pd.notnull(end.get("total_liabilities", np.nan)) and pd.notnull(eq_end):
            r["부채비율(%)"] = end["total_liabilities"] / eq_end * 100
        rows.append(r)
    return pd.DataFrame(rows).set_index("연도") if rows else pd.DataFrame()


def company_summary(ticker: str,
                    n_quarters: int = 20,
                    save: str = None,          # None | "xlsx" | "csv"
                    output_dir: str = "./dart_summary_out",
                    show: bool = True) -> dict:
    """
    티커 하나의 요약 재무분석 일괄 생성.
    반환: {"annual", "quarterly", "turnover", "dupont", "raw"} DataFrame dict
    save="xlsx" → {output_dir}/{ticker}_summary_{YYYYMMDD}.xlsx (시트 5개)
    save="csv"  → {output_dir}/{ticker}_*.csv 5개 파일
    """
    tk = norm_ticker(ticker)
    w = _prep_wide(tk)

    tables = {
        "annual":    build_annual_summary(w),
        "quarterly": build_quarterly_summary(w).tail(n_quarters),
        "turnover":  build_turnover_ttm(w).tail(n_quarters),
        "dupont":    build_dupont_ttm(w).tail(n_quarters),
        "raw":       w.tail(n_quarters),
    }

    if show:
        pd.set_option("display.width", 220, "display.max_columns", 30)
        print("=" * 90)
        print(f"■ {tk} 요약 재무분석 (최근 {n_quarters}분기)")
        print("=" * 90)
        print("\n[① 연간 요약]")
        print(tables["annual"].round(2).to_string())
        print("\n[② 분기 요약 — 최근 8분기]")
        print(tables["quarterly"].tail(8).round(2).to_string())
        print("\n[③ 회전일수 TTM]")
        t = tables["turnover"].copy()
        for c in ("매출 TTM", "COGS TTM"):
            t[c] = t[c].map(lambda v: f"{v:,.0f}" if pd.notnull(v) else "")
        print(t.round(1).to_string())
        print("\n[④ DuPont TTM — 최근 8분기]")
        print(tables["dupont"].tail(8).round(2).to_string())

    if save:
        os.makedirs(output_dir, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d")
        if save == "xlsx":
            path = os.path.join(output_dir, f"{tk}_summary_{stamp}.xlsx")
            sheet_names = {"annual": "①연간요약", "quarterly": "②분기요약",
                           "turnover": "③회전일수TTM", "dupont": "④DuPont",
                           "raw": "⑤원데이터(원)"}
            with pd.ExcelWriter(path, engine="openpyxl") as xw:
                for k, df in tables.items():
                    df.to_excel(xw, sheet_name=sheet_names[k],
                                index=(k in ("annual", "raw")))
            print(f"\n💾 저장 완료: {path}")
        elif save == "csv":
            for k, df in tables.items():
                path = os.path.join(output_dir, f"{tk}_{k}_{stamp}.csv")
                df.to_csv(path, index=(k in ("annual", "raw")),
                          encoding="utf-8-sig")
            print(f"\n💾 CSV 5개 저장 완료: {output_dir}/{tk}_*_{stamp}.csv")
        else:
            raise ValueError("save 는 None, 'xlsx', 'csv' 중 하나")

    return tables


## Cell 4 · 실행

In [24]:
# ═══════════════════════════════════════════════════════════════
#  실행
# ═══════════════════════════════════════════════════════════════
tables = company_summary(
    TICKER,
    n_quarters=N_QUARTERS,
    save=SAVE_FORMAT,
    output_dir=OUTPUT_DIR,
    show=True,
)

# 여러 종목 일괄 추출이 필요하면:
# for tk in ["005930", "000660", "035420"]:
#     company_summary(tk, n_quarters=N_QUARTERS, save="xlsx",
#                     output_dir=OUTPUT_DIR, show=False)


C:\Users\82108\AppData\Local\Temp\ipykernel_16352\3414039401.py:187: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


TypeError: unsupported operand type(s) for /: 'NoneType' and 'float'

In [20]:
tables

{'annual':          매출액(억원)   영업이익(억원)  당기순이익(억원)   영업이익률(%)    순이익률(%)     ROE(%)    부채비율(%)
 연도                                                                                
 2016  2018667.45  292406.72    3104.37  14.485136   0.153783   0.166891  35.867643
 2017  2395753.76  536450.38    8421.78  22.391716   0.351529   0.413385  40.682587
 2018  2437714.15  588866.69    4539.80  24.156511   0.186232   0.196424  36.973922
 2019  2304008.81  277685.09  215050.54  12.052258   9.333755   8.422890  34.115921
 2020  2368069.88  359938.76  260908.46  15.199668  11.017769   9.684287  37.067743
 2021  2796047.99  516338.56  392437.91  18.466727  14.035450  13.512587  39.921697
 2022  3022313.60  433766.30  547300.18  14.352127  18.108650  16.593665  26.405922
 2023  2589354.94   65669.76  144734.01   2.536144   5.589578   4.029189  25.359837
 2024  3008709.03  327259.61       0.00  10.877077   0.000000   0.000000  27.931898
 2025  3336059.38  436010.51   80284.07  13.069627   2.406554   1.